# Emotion Recognition Using Audio Features

In [ ]:
from huggingface_hub import snapshot_download
from IPython.display import Audio
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
import os, csv
import pandas as pd
import torch
import torchaudio
import torchaudio.functional as AF
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import make_circles, make_classification, make_moons
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay

from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from sklearn.metrics import classification_report

In [ ]:
EMO2IDX = {"angry": 0, "happy": 1, "neutral": 2, "sad": 3}
IDX2EMO = {idx: emo for emo, idx in EMO2IDX.items()}

ROOT = snapshot_download("OlhaHavryliuk/UA-SER", repo_type="dataset")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cpu")

sample_rate = 16000
n_epochs = 5

In [ ]:
class UASER(Dataset):
    def __init__(self, split):
        self.dir = os.path.join(ROOT, "clips")
        self.rows = [r for r in csv.DictReader(open(os.path.join(ROOT, "dataset.csv")))
                     if r["split"] == split]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        wav, _ = sf.read(os.path.join(self.dir, r["filename"]), dtype="float32")
        return torch.from_numpy(wav), EMO2IDX[r["emotion"]]


def collate(batch):
    wavs, labels = zip(*batch)
    T = max(w.shape[0] for w in wavs)
    audio = torch.zeros(len(wavs), T)
    for i, w in enumerate(wavs):
        audio[i, : w.shape[0]] = w
    return audio, torch.tensor(labels)


train_loader = DataLoader(UASER("train"), batch_size=16, shuffle=True, collate_fn=collate)
test_loader  = DataLoader(UASER("test"),  batch_size=16, collate_fn=collate)


if __name__ == "__main__":
    print("train:", len(train_loader.dataset), "| test:", len(test_loader.dataset))
    audio, labels = next(iter(train_loader))
    print(audio.shape, labels.tolist())

In [ ]:
def metrics(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    uar = balanced_accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    return [accuracy, uar, macro_f1]

In [ ]:
def to_np(loader, target_len):
    all_x, all_y = [], []

    for x, y in loader:
        x, y = x.detach().cpu(), y.detach().cpu()

        for i in range(x.shape[0]):
            sample = x[i].reshape(-1).numpy()
            sample = np.asarray(sample, dtype=np.float32)

            if len(sample) > target_len:
                sample = sample[:target_len]
            elif len(sample) < target_len:
                sample = np.pad(
                    sample,
                    (0, target_len - len(sample)),
                    mode="constant"
                )

            all_x.append(sample)

        if y.ndim > 1:
            y = torch.argmax(y, dim=1)

        all_y.append(y.numpy())

    all_x = np.stack(all_x)
    all_y = np.concatenate(all_y, axis=0)
    return all_x, all_y


target_len = sample_rate * 3  # 3 seconds

x_train, y_train = to_np(train_loader, target_len)
x_test, y_test = to_np(test_loader, target_len)

In [ ]:
# model_wav2vec2 = AutoModelForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
# model_emo2vec = AutoModelForCTC.from_pretrained("iic/emotion2vec_plus_large").to(DEVICE)

# models = [model_wav2vec2, model_emo2vec]
# models_name = ["wav2vec2-xls-r-300m-uk", "emotion2vec+"]

In [ ]:
models = [{
        "name": "wav2vec2",
        "model": "robinhad/wav2vec2-xls-r-300m-uk",
    },
    {
        "name": "data2vec",
        "model": "facebook/data2vec-audio-base",
    }]

In [ ]:
audio1, label1 = audio[0], labels[0]

In [ ]:
trained_models = {}

for approach in models:
    feature_extractor = AutoFeatureExtractor.from_pretrained(approach["model"])

    model = AutoModelForAudioClassification.from_pretrained(
        approach["model"],
        num_labels=4,
        label2id=EMO2IDX,
        id2label=IDX2EMO,
        ignore_mismatched_sizes=True
    ).to(DEVICE)
    
    opt = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_curve = []
    
    for epoch in range(1, n_epochs+1):
        model.train()
        
        print(f"Epoch: {epoch}/{n_epochs}")
        running_loss = 0.0
        
        for x, y in train_loader: # audio, labels
            if x.ndim == 3:
                x = x.squeeze(1)
            x = x.float().cpu().numpy()
            y = y.to(DEVICE)
            
            inputs = feature_extractor(
                list(x),
                sampling_rate=sample_rate,
                return_tensors="pt",
                padding=True
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
                        
            logits = model(**inputs, labels=y)
            loss = logits.loss
    
            opt.zero_grad()
            loss.backward
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step() 

            running_loss += loss.item()
                
        loss_curve = np.append(loss_curve, running_loss)

    torch.save(model.state_dict(), f'{approach["name"]}.pth')
    print(f"Model: {approach['name']} saved!")

    trained_models[approach["name"]] = {
        "model": model,
        "feature_extractor": feature_extractor,
    }
    
    plt.plot(loss_curve)
    plt.title(approach["name"])
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

In [ ]:
result = {}

@torch.no_grad()
for approach in trained_models:
    result[approach["name"]] = {}

    feature_extractor = approach["feature_extractor"]
    model = approach["model"]
    model.eval()

    y_true = []
    y_pred = []
    
    for x, y in zip(audio1, label1):
        if x.ndim == 3:
            x = x.squeeze(1)

        x = x.float().cpu().numpy()

        inputs = feature_extractor(
            list(x),
            sampling_rate=sample_rate,
            return_tensors="pt",
            padding=True
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    list_metrics = metrics(y_true, y_pred)    
    score = metrics(preds, y_test)
        
    result[approach["name"]] = {
            "accuracy": score[0],
            "uar": score[1],
            "macro_f1": score[2],
        }
result

In [ ]:
df_result = pd.DataFrame(result).T
df_result.to_csv("Self-Supervised_metrics.csv", index=False)
df_result